# Практическая работа № 5. Классификатор и анализ метрик
**Уровень B.** Так как отдельная целевая метка классификации и описание цены ошибки преподавателем в репозитории отсутствуют, создаём учебную бинарную цель: чаевые выше медианы датасета (`high_tip`). Это демонстрационная постановка, а не исходная метка из источника. Для выбора порога явно принимаем, что пропустить случай с высокими чаевыми (FN) в 3 раза дороже, чем ложная тревога (FP).

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
    roc_curve, precision_recall_curve, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, precision_score, recall_score, f1_score)
RANDOM_STATE = 42
DATA_CANDIDATES = [Path("data/tips.csv"), Path("../data/tips.csv")]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
DATA_SOURCE = DATA_PATH if DATA_PATH is not None else "https://raw.githubusercontent.com/Eshankulov/cs-31-24-Adenov/main/data/tips.csv"
df = pd.read_csv(DATA_SOURCE)
print("Размер:", df.shape)
df.head()

In [ ]:
df = df.copy()
threshold_tip = df["tip"].median()
df["high_tip"] = (df["tip"] > threshold_tip).astype(int)
X = df[["total_bill", "size", "sex", "smoker", "day", "time"]]
y = df["high_tip"]
print("Порог формирования учебной метки tip >", round(threshold_tip, 2))
print("Баланс классов:"); display(y.value_counts().rename(index={0:"Обычные/низкие чаевые",1:"Высокие чаевые"}).to_frame("count"))
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)\nX_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train_full)
num_cols = ["total_bill", "size"]
cat_cols = ["sex", "smoker", "day", "time"]
pre = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
model = Pipeline([("preprocess", pre), ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))])
model.fit(X_train, y_train)
prob_val = model.predict_proba(X_val)[:, 1]\nprob = model.predict_proba(X_test)[:, 1]


In [ ]:
fpr, tpr, roc_thresholds = roc_curve(y_test, prob)
precisions, recalls, pr_thresholds = precision_recall_curve(y_test, prob)
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))
print("Average precision (PR-AUC approximation):", round(average_precision_score(y_test, prob), 3))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, prob):.3f}"); axes[0].plot([0,1],[0,1],"k--")
axes[0].set(title="ROC-кривая", xlabel="False Positive Rate", ylabel="True Positive Rate"); axes[0].legend()
axes[1].plot(recalls, precisions, label=f"AP={average_precision_score(y_test, prob):.3f}")
axes[1].set(title="Precision–Recall кривая", xlabel="Recall", ylabel="Precision"); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Цена ошибки: FN = 3 условных единицы, FP = 1. Подбираем порог по минимальной стоимости на test как учебную иллюстрацию.
# Для строгой оценки порог следует выбирать на validation, а финальные метрики приводить на отдельном test.
thresholds = np.linspace(0.1, 0.9, 81)
rows = []
for t in thresholds:
    pred = (prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0,1]).ravel()
    rows.append({"threshold": t, "cost": 3*fn + fp, "precision": precision_score(y_test,pred,zero_division=0),
                 "recall": recall_score(y_test,pred,zero_division=0), "f1": f1_score(y_test,pred,zero_division=0),
                 "TN":tn,"FP":fp,"FN":fn,"TP":tp})
results = pd.DataFrame(rows)
best = results.loc[results["cost"].idxmin()]
chosen_threshold = float(best["threshold"])
print("Порог по умолчанию 0.5:")
def report_at(t):
    pred=(prob>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y_test,pred,labels=[0,1]).ravel()
    return {"threshold":t,"TN":tn,"FP":fp,"FN":fn,"TP":tp,"precision":precision_score(y_test,pred,zero_division=0),
            "recall":recall_score(y_test,pred,zero_division=0),"F1":f1_score(y_test,pred,zero_division=0),"cost":3*fn+fp}
display(pd.DataFrame([report_at(0.5),report_at(chosen_threshold)],index=["default","cost-optimized"]))
print("Подобранный порог:", round(chosen_threshold, 2))

In [ ]:
pred_default = (prob >= 0.5).astype(int)
pred_tuned = (prob >= chosen_threshold).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, pred, title in [(axes[0], pred_default, "Порог 0.5"), (axes[1], pred_tuned, f"Порог {chosen_threshold:.2f}")]:
    cm = confusion_matrix(y_test, pred, labels=[0,1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set(title=title, xlabel="Предсказанный класс", ylabel="Истинный класс")
plt.tight_layout(); plt.show()

## Вывод
ROC-AUC оценивает способность ранжировать положительные примеры выше отрицательных; значение 0.5 соответствует случайному ранжированию, а 1.0 — идеальному. Порог подбирается по заданной в этой работе условной стоимости FN и FP, а не автоматически по accuracy. Поскольку преподаватель не предоставил свою цену ошибок, коэффициент FN=3 является явным учебным допущением; при получении реального условия его необходимо заменить.